In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
# ============================================================
# Submission dtype helpers
# ============================================================
# These wrappers reduce DataFrame memory use without changing the financial
# calculations: identifiers/counters are downcast, repeated labels become
# categoricals, and continuous numerical columns remain float64.
import numpy as np

_PD_READ_CSV = pd.read_csv
_PD_READ_EXCEL = pd.read_excel

_INTEGER_DTYPE_CANDIDATES = {
    "match_id": np.int32,
    "hour": np.int16,
    "hour_index": np.int16,
    "replication": np.int16,
    "case_order": np.int16,
    "enabled": np.int8,
    "rank": np.int32,
}

_CATEGORY_DTYPE_CANDIDATES = {
    "case_id",
    "case_family",
    "case_label",
    "combined_category",
    "metric",
    "mutation_axis",
    "mutation_direction",
    "mutation_family",
    "mutation_label",
    "ppa_type",
    "profile_type",
    "risk_group",
    "risk_label",
    "scenario_name",
    "scenario_type",
    "solution_type",
    "status",
    "variable",
    "var_i",
    "var_j",
}


def _integer_dtype_fits(values, dtype) -> bool:
    if len(values) == 0:
        return True
    info = np.iinfo(dtype)
    return float(np.nanmin(values)) >= info.min and float(np.nanmax(values)) <= info.max


def optimize_dataframe_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """Conservatively compact non-financial columns after file loading."""
    if not isinstance(df, pd.DataFrame) or df.empty:
        return df

    for col in df.columns:
        series = df[col]
        if pd.api.types.is_integer_dtype(series.dtype):
            df[col] = pd.to_numeric(series, downcast="integer")

    for col, dtype in _INTEGER_DTYPE_CANDIDATES.items():
        if col not in df.columns:
            continue
        numeric = pd.to_numeric(df[col], errors="coerce")
        if numeric.isna().any():
            continue
        values = numeric.to_numpy(dtype="float64", copy=False)
        rounded = np.rint(values)
        if np.array_equal(values, rounded) and _integer_dtype_fits(rounded, dtype):
            df[col] = rounded.astype(dtype, copy=False)

    n_rows = len(df)
    for col in _CATEGORY_DTYPE_CANDIDATES.intersection(df.columns):
        series = df[col]
        if pd.api.types.is_categorical_dtype(series.dtype):
            continue
        if not (pd.api.types.is_object_dtype(series.dtype) or pd.api.types.is_string_dtype(series.dtype)):
            continue
        non_null = series.dropna()
        if non_null.empty:
            continue
        n_unique = int(non_null.nunique())
        if n_unique <= min(128, max(2, n_rows // 2)):
            df[col] = series.astype("category")

    return df


def read_csv_optimized(*args, **kwargs) -> pd.DataFrame:
    return optimize_dataframe_dtypes(_PD_READ_CSV(*args, **kwargs))


def read_excel_optimized(*args, **kwargs) -> pd.DataFrame:
    return optimize_dataframe_dtypes(_PD_READ_EXCEL(*args, **kwargs))


# Help function

In [ ]:
def process_demand_group(group_df):
    # Extract identifiers (assuming consistent within group)
    market_region = group_df['mkt_region'].iloc[0]
    zone = group_df['zone'].iloc[0]
    load_area = group_df['load_area'].iloc[0]
    
    # # Convert 'is_verified' to boolean if necessary
    # if group_df['is_verified'].dtype == 'object':
    #     group_df['is_verified'] = group_df['is_verified'].map({'true': True, 'false': False, True: True, False: False})
    
    # # Filter for verified=True
    # group_df = group_df[group_df['is_verified'] == True].copy()
    
    if group_df.empty:
        return None  # Skip if no verified data
    
    # Parse the time using pd.to_datetime (handles strings or numerics)
    group_df['datetime'] = pd.to_datetime(group_df['datetime_beginning_ept'], errors='coerce')
    
    # Drop any rows where datetime parsing failed
    group_df = group_df.dropna(subset=['datetime'])
    
    group_df['hour'] = group_df['datetime'].dt.hour
    group_df['month'] = group_df['datetime'].dt.month
    
   # Divide the data into 4 parts based on meteorological season (Q1: 12,1,2; Q2: 3-5; Q3: 6-8; Q4: 9-11)
    season_mapping = {12: 'Q1', 1: 'Q1', 2: 'Q1', 3: 'Q2', 4: 'Q2', 5: 'Q2', 6: 'Q3', 7: 'Q3', 8: 'Q3', 9: 'Q4', 10: 'Q4', 11: 'Q4'}
    group_df['part'] = group_df['month'].map(season_mapping)
    
    # Calculate Overall Statistics for 'mw'
    overall_mean = group_df['mw'].mean()
    overall_50th = group_df['mw'].median() 
    overall_var = group_df['mw'].var()
    overall_5th = group_df['mw'].quantile(0.05)
    overall_95th = group_df['mw'].quantile(0.95)
    
    # Initialize the row structure
    result = {
        'Market_Region': market_region,
        'Zone': zone,
        'Load_Area': load_area,
        'Overall_Mean': overall_mean,
        'Overall_50th': overall_50th,
        'Overall_Variance': overall_var,
        'Overall_5th': overall_5th,
        'Overall_95th': overall_95th
    }
    
    # Calculate Hourly Statistics divided by the 4 parts
    hourly_stats = group_df.groupby(['part', 'hour'])['mw'].agg(
        mean='mean', 
        median='median', 
        var='var',
        p5=lambda x: x.quantile(0.05),
        p95=lambda x: x.quantile(0.95)
    ).reset_index()
    
    # Flatten the hourly results into wide columns
    for _, row in hourly_stats.iterrows():
        part = row['part']
        hour = int(row['hour'])
        
        result[f'{part}_H{hour}_Mean'] = row['mean']
        result[f'{part}_H{hour}_50th'] = row['median']
        result[f'{part}_H{hour}_Variance'] = row['var']
        result[f'{part}_H{hour}_5th'] = row['p5']
        result[f'{part}_H{hour}_95th'] = row['p95']
        
    return result

# Results

In [ ]:
# Manually define the file path here
file_path = 'Original Data/Demand/hourly_metered_load_all.csv'

# Read the CSV file
df = read_csv_optimized(file_path)

# Process each unique load_area
unique_load_areas = df['load_area'].unique()
all_results = []

for load_area in unique_load_areas:
    group_df = df[df['load_area'] == load_area].copy()  # Copy to avoid SettingWithCopyWarning
    if not group_df.empty:
        result = process_demand_group(group_df)
        if result is not None:
            all_results.append(result)

# Combine into DataFrame and export
final_summary_df = pd.DataFrame(all_results)

# Output file
output_filename = 'Demand_Hourly Summary.xlsx'

output_dir = 'Processed Data'
output_path = os.path.join(output_dir, output_filename)

final_summary_df.to_excel(output_path, index=False)

print(f"Data saved to {output_path} with {len(final_summary_df.columns)} columns.")

# Plotting

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors

# File reading part (adjust path if needed)
file_path = './Processed Data/Demand_Hourly Summary.xlsx'
df = read_excel_optimized(file_path)

# Manually choose the load area for plotting
load_area_to_plot = 'DUQ'  # Replace with the actual load area, e.g., 'AECO'

# Filter the row for the selected load area
row = df[df['Load_Area'] == load_area_to_plot].iloc[0]  # Assumes unique load areas

# Create the figure with 4 subfigures (2x2 grid)
fig, axs = plt.subplots(2, 2, figsize=(15, 10), sharey=True)  # Share y-axis for consistency
axs = axs.flatten()  # Flatten for easy indexing

quarters = ['Q1', 'Q2', 'Q3', 'Q4']

for i, quarter in enumerate(quarters):
    ax = axs[i]
    
    # Collect stats for each hour
    stats = []
    means = []
    for hour in range(24):
        mean = row[f'{quarter}_H{hour}_Mean']
        median = row[f'{quarter}_H{hour}_50th']
        fifth = row[f'{quarter}_H{hour}_5th']
        ninetyfifth = row[f'{quarter}_H{hour}_95th']
        
        # Custom stats for boxplot: box from 5th to 95th, median line
        stat = {
            'med': median,
            'q1': fifth,
            'q3': ninetyfifth,
            'whislo': fifth,      # Whiskers coincide with box ends (effectively no visible whiskers)
            'whishi': ninetyfifth,
            'fliers': []          # No outliers
        }
        stats.append(stat)
        means.append(mean)
    
    # Plot the boxplots (boxes from 5th to 95th with median)
    ax.bxp(stats, positions=range(24), widths=0.6, showfliers=False, patch_artist=True,
           boxprops=dict(facecolor='lightblue', color='sandybrown'),
           medianprops=dict(color='black', linewidth=2))
    
    # Overlay the means as red points
    ax.scatter(range(24), means, color='red', label='Mean', zorder=3, s=25)
    
    # Set labels and title
    ax.set_xticks(range(24))
    ax.set_xlabel('Hour')
    if i % 2 == 0:  # Only set y-label for left subplots (Q1 and Q3)
        ax.set_ylabel('Demand (MW)')
    ax.set_title(f'{quarter} - Load Area: {load_area_to_plot}')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.7)

# Adjust layout and show the plot
plt.tight_layout()
plt.show()